<div dir="rtl" style="text-align: right; font-family: Tahoma, Vazirmatn, sans-serif; line-height: 1.8;">

<h1>تکلیف: پیاده‌سازی سیستم RAG و مقایسه استراتژی‌های Prompting</h1>

<h2>مقایسه استراتژی‌های Zero-shot، Few-shot و Chain-of-Thought در سیستم RAG</h2>

<hr>

<h2>راه‌های ارتباطی</h2>
<ul>
  <li>📧 <b>ایمیل:</b> <a href="mailto:erfanshahabi@ut.ac.ir">erfanshahabi@ut.ac.ir</a></li>
</ul>

<hr>

<h2>مجموعه‌داده (Dataset)</h2>

<p>
در این تکلیف از زیر مجموعه‌ای از مجموعه‌داده
<b>rajpurkar/squad</b>
و از معیارهای ارزیابی
<b>ROUGE (Recall-Oriented Understudy for Gisting Evaluation)</b>
و
شباهت معنایی
استفاده می‌کنیم.
</p>

<ul>
  <li>از مجموعه‌داده اصلی <b>SQuAD</b>، یک زیرمجموعه (<b>Subset</b>) انتخاب شده است:
    <ul>
      <li><b>100</b> پاراگراف (<b>Context</b>) از ویکی‌پدیا به عنوان corpus — فایل <b>squad_corpus_100.csv</b></li>
      <li><b>20</b> سوال و جواب (<b>Gold QA</b>) برای ارزیابی — فایل <b>squad_gold_20.csv</b></li>
      <li><b>4561</b> نمونه برای Fine-tuning — فایل <b>finetune_train.csv</b></li>
      <li><b>303</b> نمونه برای اعتبارسنجی Fine-tuning — فایل <b>finetune_val.csv</b></li>
    </ul>
  </li>
  <li>هر نمونه شامل:
    <ul>
      <li><b>context</b>: پاراگراف متنی از ویکی‌پدیا</li>
      <li><b>question</b>: سوال مرتبط با متن</li>
      <li><b>answer</b>: جواب gold به صورت span از متن</li>
    </ul>
  </li>
  <li>تمامی فایل‌های داده در فایل تمرین قرار داده شده‌اند.</li>
</ul>

<hr>

<h2>مدل‌ها (Models)</h2>

<p>
<b>Qwen/Qwen2.5-1.5B-Instruct, FacebookAI/roberta-large-mnli, BAAI/bge-small-en-v1.5</b>
</p>

<hr>

<h2>نکات مهم</h2>

<ul>
  <li>در پایان <b>هر مرحله</b>، نتایج را در یک فایل <b>CSV</b> ذخیره کنید و در فایل ارسالی پاسخ تمرین قرار دهید.</li>
  <li>فایل‌های CSV نتایج پاسخ مدل به سوالات در هر مرحله باید در فایل ارسالی پاسخ تمرین قرار داده شوند.</li>
  <li>فایل نهایی ارسالی باید یک فایل ZIP شامل همین نوت بوک و فایل پاسخ‌های مدل در هر مرحله در قال CSV باشد.</li>
  <li>مجموع نمرات این تمرین <b>57</b> نمره می‌باشد.</li>
</ul>

</div>

---
## 0  Install Requirements and import libraries and datasets

#### Amirhossein Rahati 810104303

In [ ]:
!pip install --upgrade pip setuptools wheel

!pip install \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu128

!pip install \
    transformers \
    accelerate \
    datasets \
    sentence-transformers \
    peft \
    torchao \
    lancedb \
    rouge-score \
    numpy \
    pandas \
    scipy \
    huggingface_hub \
    tokenizers \
    safetensors \
    ipykernel

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM,AutoModelForSequenceClassification ,TrainingArguments, Trainer, DataCollatorForLanguageModeling
import accelerate
from sentence_transformers import SentenceTransformer
import lancedb
from peft import LoraConfig, PeftModel, get_peft_model , prepare_model_for_kbit_training
from rouge_score import rouge_scorer
from scipy.special import softmax
from datasets import Dataset
import gc
import re
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
NLI_MODEL_NAME = "FacebookAI/roberta-large-mnli"
MODEL_NAME = "Qwen/Qwen3.5-2B"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"


cuda


---
## 1  Define Schema and Store Embeddings in LanceDB (3 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to define an appropriate Schema for storing text passages in LanceDB and use the embedding model
<b>BAAI/bge-small-en-v1.5</b>
to convert the passages from <b>squad_corpus_100.csv</b> into vector representations and store them in the database.
Then, run a sample semantic search query and display the top results to verify that the retrieval is working correctly.
</p>

In [2]:
DATA_DIR = Path("./Data")

corpus_df = pd.read_csv(DATA_DIR / "squad_corpus_100.csv")
gold_df = pd.read_csv(DATA_DIR / "squad_gold_20.csv")

print(f"Corpus: {len(corpus_df)} - gold: {len(gold_df)}")

Corpus: 100 - gold: 20


In [3]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

db = lancedb.connect("./lancedb_data")
table_name = "corpus"

if table_name in db.table_names():
    db.drop_table(table_name)

embeddings = embedding_model.encode(
    corpus_df["context"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=False
)

data = []
for idx, (_, row) in enumerate(corpus_df.iterrows()):
    data.append({
        "id": idx,
        "text": row["context"],
        "vector": embeddings[idx].tolist()
    })

table = db.create_table(table_name, data=data)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_21325/3082609816.py:6: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name in db.table_names():


---
## 2  Define a function for semantic search in lanceDB (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to define a retrieval function that searches LanceDB using the questions from <b>squad_gold_20.csv</b>.
For each question, the function should return the top <b>5</b> most semantically similar passages from the database.
</p>

In [4]:
def semantic_search(query, table, embedding_model, top_k=5):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True,show_progress_bar=False)[0]
    results = table.search(query_embedding.tolist()).limit(top_k).to_pandas()
    return [
        {"text": row["text"], "distance": row["_distance"]}
        for _, row in results.iterrows()
    ]


---
## 3  Loading Model (2 Points)

Load the **Qwen/Qwen3.5-2B** model and its tokenizer from 🤗 Hugging Face using `AutoModelForCausalLM` and `AutoTokenizer`.

In [5]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True,padding_side="left")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained( MODEL_NAME, dtype=torch.float16, device_map="auto",
                                             trust_remote_code=True, low_cpu_mem_usage=True)

print("parameter count:" ,  sum(p.numel() for p in model.parameters()))

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

parameter count: 1881825088


In [ ]:
def clean_qwen_response(full_response, prompt):
    response = full_response.replace(prompt, "").strip()
    
    response = re.sub(r'<\|im_start\|>.*?<\|im_end\|>', '', response, flags=re.DOTALL)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    response = re.sub(r'<\|endoftext\|>', '', response)
    response = response.replace("assistant", "").strip()
    response = response.replace("Assistant", "").strip()
    
    lines = [line for line in response.split('\n') if line.strip() and not line.strip().startswith('system') and not line.strip().startswith('user')]
    response = ' '.join(lines).strip()
    return response

def generate_response(prompt, model, tokenizer, max_new_tokens=50, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            top_p=0.9
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_qwen_response(full_response, prompt)
    # return full_response

---
## 4  Search Without RAG (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, define a prompt-building function and an appropriate system prompt, then answer the 20 questions from <b>squad_gold_20.csv</b> using only the model's internal knowledge, without retrieving any passages from LanceDB.
Store the predicted answers alongside the gold answers for evaluation in later sections.
You have to print the model responses.
</p>

In [34]:
def create_no_rag_prompt(question):
    system_prompt = """You are a helpful assistant. Answer the following question accurately and concisely.
IMPORTANT: Provide only a SHORT, CONCISE answer with entire response as MAXIMUM 20 WORDS. Do not add explanations or context."""
    
    prompt = f"""<|im_start|>system
{system_prompt}
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""
    return prompt

def answer_without_rag(question, model, tokenizer):
    prompt = create_no_rag_prompt(question)
    return generate_response(prompt, model, tokenizer, max_new_tokens=200, temperature=0.1)


no_rag_results = []

for idx, row in gold_df.iterrows():
    question = row["question"]
    gold_answer = row["answer"]
    
    predicted_answer = answer_without_rag(question, model, tokenizer).split(question)[1].strip()
    
    print(f"[{idx+1}] Q: {question}")
    print(f"Gold: {gold_answer}")
    print(f"Pred: {predicted_answer}\n")
    print("------\n")
    no_rag_results.append({
        "id": row["id"],
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": predicted_answer
    })

pd.DataFrame(no_rag_results).to_csv("no_rag_predictions.csv", index=False)

[1] Q: When is the last time a fumble return touchdown happened in a Super Bowl?
Gold: Super Bowl XXVIII
Pred: The last fumble return touchdown occurred in Super Bowl LII (51) in 2017.

------

[2] Q: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Gold: last week of February 1974,
Pred: AAA reported that approximately 15% of US gas stations ran out of gasoline in recent years.

------

[3] Q: What choice did French have for surrendering land?
Gold: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Pred: French had no choice but to surrender land after losing all its colonies in 1763.

------

[4] Q: Which articles state that the member states' rights to deliver public services may not be obstructed?
Gold: Articles 106 and 107
Pred: Articles 15(3) of the Treaty on European Union and Article 4(3) of the Treaty on European Community

------

[5] Q: The time required to output an 

---
## 5 RAG - ZeroShot (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Zero-shot</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that includes the retrieved context.
The model should answer the question based only on the provided passages, without any examples or reasoning steps.
Store the predicted answers for evaluation in later sections.
You have to print the model responses.
</p>

In [18]:
def create_zero_shot_prompt(question, contexts):
    system_prompt = """You are a helpful assistant. Answer the question based ONLY on the provided contexts.
Do not use your prior knowledge. If the answer is not in the contexts, say "I don't know".
IMPORTANT: Provide only a SHORT, CONCISE answer with entire response as MAXIMUM 20 WORDS. Do not add explanations or context."""
    
    context_text = "\n\n".join([f"Context {i+1}: {ctx['text']}" for i, ctx in enumerate(contexts)])
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Contexts:\n{context_text}\n\nQuestion: {question}"}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True
    )
    return prompt


def answer_zero_shot(question, table, embedding_model, model, tokenizer, top_k=5):
    contexts = semantic_search(question, table, embedding_model, top_k)
    prompt = create_zero_shot_prompt(question, contexts)
    response = generate_response(prompt, model, tokenizer, max_new_tokens=200, temperature=0.1)
    return response, contexts


zero_shot_results = []

for idx, row in gold_df.iterrows():
    question = row["question"]
    gold_answer = row["answer"]
    
    predicted_answer, contexts = answer_zero_shot(question, table, embedding_model, model, tokenizer)
    predicted_answer = predicted_answer.split(question)[1].strip()
    
    print(f"[{idx+1}] Q: {question}")
    print(f"Gold: {gold_answer}")
    print(f"Pred: {predicted_answer}\n")
    print("------\n")
    
    zero_shot_results.append({
        "id": row["id"],
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": predicted_answer,
        "retrieved_contexts": [ctx["text"] for ctx in contexts]
    })

pd.DataFrame(zero_shot_results).to_csv("rag_zero_shot_predictions.csv", index=False)

[1] Q: When is the last time a fumble return touchdown happened in a Super Bowl?
Gold: Super Bowl XXVIII
Pred: It was in Super Bowl XXXVI in 2002.

------

[2] Q: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Gold: last week of February 1974,
Pred: 20 percent of American gasoline stations had no fuel.

------

[3] Q: What choice did French have for surrendering land?
Gold: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Pred: France chose to cede its continental North American possessions east of the Mississippi while retaining Saint Pierre and Miquelon.

------

[4] Q: Which articles state that the member states' rights to deliver public services may not be obstructed?
Gold: Articles 106 and 107
Pred: Article 106 states that member states' rights to deliver public services may not be obstructed.

------

[5] Q: The time required to output an answer on a deterministic Turin

---
## 6 RAG - FewShot (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Few-shot</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that includes the retrieved context along with <b>3 examples</b> of question-answer pairs.
The examples should guide the model on the expected format and style of the answer.
Store the predicted answers for evaluation in later sections.
You have to print the model responses.
</p>

In [33]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "When did the American Civil War end?",
        "answer": "April 9, 1865"
    },
    {
        "question": "What is the official language of Brazil?",
        "answer": "Portuguese"
    },
    {
        "question": "What are the main causes of climate change?",
        "answer": "Burning fossil fuels, deforestation, and industrial emissions"
    }
]

def create_few_shot_prompt(question, contexts, examples=FEW_SHOT_EXAMPLES):
    system_prompt = """You are a helpful assistant. Answer the question based ONLY on the provided contexts.
The examples below ONLY demonstrate the expected OUTPUT FORMAT and STYLE. Do NOT use the information from the examples to answer the question.
IMPORTANT: Provide only a SHORT, CONCISE answer with entire response as MAXIMUM 20 WORDS. Do not add explanations or context."""
    
    context_text = "\n\n".join([f"Context {i+1}: {ctx['text']}" for i, ctx in enumerate(contexts)])
    
    examples_text = "\n\n".join([
        f"Example {i+1}:\nQuestion: {ex['question']}\nAnswer: {ex['answer']}"
        for i, ex in enumerate(examples)
    ])
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"OUTPUT FORMAT EXAMPLES (ignore the content, only follow the format):\n\n{examples_text}\n\nNow answer the following question using ONLY the contexts provided below.\n\nContexts:\n{context_text}\n\nQuestion: {question}"}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True
    )
    return prompt


def answer_few_shot(question, table, embedding_model, model, tokenizer, top_k=5):
    contexts = semantic_search(question, table, embedding_model, top_k)
    prompt = create_few_shot_prompt(question, contexts)
    response = generate_response(prompt, model, tokenizer, max_new_tokens=200, temperature=0.1)
    return response, contexts


few_shot_results = []

for idx, row in gold_df.iterrows():
    question = row["question"]
    gold_answer = row["answer"]
    
    predicted_answer, contexts = answer_few_shot(question, table, embedding_model, model, tokenizer)
    predicted_answer = predicted_answer.split(question)[-1].strip()
    
    print(f"[{idx+1}] Q: {question}")
    print(f"Gold: {gold_answer}")
    print(f"Pred: {predicted_answer}\n")
    print("------\n")
    
    few_shot_results.append({
        "id": row["id"],
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": predicted_answer,
        "retrieved_contexts": [ctx["text"] for ctx in contexts]
    })

pd.DataFrame(few_shot_results).to_csv("rag_few_shot_predictions.csv", index=False)

[1] Q: When is the last time a fumble return touchdown happened in a Super Bowl?
Gold: Super Bowl XXVIII
Pred: Super Bowl XXVIII ended in 1993.

------

[2] Q: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Gold: last week of February 1974,
Pred: 20%

------

[3] Q: What choice did French have for surrendering land?
Gold: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Pred: France chose to cede the continental North American possessions east of the Mississippi.

------

[4] Q: Which articles state that the member states' rights to deliver public services may not be obstructed?
Gold: Articles 106 and 107
Pred: Article 106 and Article 107

------

[5] Q: The time required to output an answer on a deterministic Turing machine is expressed as what?
Gold: state transitions
Pred: It is the total number of state transitions or steps the machine makes before it halts and outputs th

---
## 7 RAG - CoT (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, implement a <b>Chain-of-Thought (CoT)</b> RAG pipeline. For each question in <b>squad_gold_20.csv</b>,
retrieve the top 5 relevant passages from LanceDB and construct a prompt that guides the model to reason step by step before giving the final answer.
Store both the full reasoning output and the final extracted answer for evaluation in later sections.
You have to print the model responses.
</p>

In [34]:
def create_cot_prompt(question, contexts):
    system_prompt = """You are a helpful assistant. Answer the question based ONLY on the provided contexts.
Think step by step before giving your final answer. Show your reasoning.
IMPORTANT: Provide only a SHORT, CONCISE answer with entire response as MAXIMUM 20 WORDS. Do not add explanations or context."""

    context_text = "\n\n".join([f"Context {i+1}: {ctx['text']}" for i, ctx in enumerate(contexts)])

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Contexts:\n{context_text}\n\nQuestion: {question}\n\nThink step by step and provide your final answer."}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True
    )
    return prompt


def extract_final_answer(full_response):
    patterns = [
        "Final answer:",
        "Answer:",
        "Therefore,",
        "So the answer is",
        "The answer is"
    ]
    
    response_lower = full_response.lower()
    
    for pattern in patterns:
        if pattern.lower() in response_lower:
            pos = response_lower.find(pattern.lower())
            answer_part = full_response[pos + len(pattern):].strip()
            lines = answer_part.split('\n')
            for line in lines:
                if line.strip():
                    return line.strip()
    
    sentences = full_response.split('.')
    if len(sentences) > 1:
        for sentence in reversed(sentences):
            if sentence.strip():
                return sentence.strip()
    return full_response


def answer_cot(question, table, embedding_model, model, tokenizer, top_k=5):
    contexts = semantic_search(question, table, embedding_model, top_k)
    prompt = create_cot_prompt(question, contexts)
    full_response = generate_response(prompt, model, tokenizer, max_new_tokens=200, temperature=0.7)
    final_answer = extract_final_answer(full_response)
    return final_answer, full_response, contexts


cot_results = []

for idx, row in gold_df.iterrows():
    question = row["question"]
    gold_answer = row["answer"]

    final_answer, reasoning, contexts = answer_cot(question, table, embedding_model, model, tokenizer)
    final_answer = final_answer.split(question + " Think step by step and provide your final answer.")[-1].strip()

    print(f"[{idx+1}] Q: {question}")
    print(f"Gold: {gold_answer}")
    print(f"Pred: {final_answer}")
    print("------\n")

    cot_results.append({
        "id": row["id"],
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": final_answer,
        "full_reasoning": reasoning,
        "retrieved_contexts": [ctx["text"] for ctx in contexts]
    })

pd.DataFrame(cot_results).to_csv("rag_cot_predictions.csv", index=False)

[1] Q: When is the last time a fumble return touchdown happened in a Super Bowl?
Gold: Super Bowl XXVIII
Pred: Based on Context 1, the last time a fumble return touchdown occurred in a Super Bowl was in 1993
------

[2] Q: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Gold: last week of February 1974,
Pred: In the last week of February 1974, 20% of American gasoline stations had no fuel according to the American Automobile Association
------

[3] Q: What choice did French have for surrendering land?
Gold: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Pred: France chose to retain Saint Pierre and Miquelon and fishing rights instead of ceding North American possessions east of the Mississippi or Caribbean islands
------

[4] Q: Which articles state that the member states' rights to deliver public services may not be obstructed?
Gold: Articles 106 and 107
Pred: Article 106 a

---
## 8 Evaluation with ROUGE (2 Points)

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, evaluate the predicted answers from all four methods (No RAG, Zero-shot, Few-shot, and CoT)
using <b>ROUGE-1</b>, <b>ROUGE-2</b>, and <b>ROUGE-L</b> metrics with the <b>rouge-score</b> library.
For each method, compute and display the <b>average score</b> across all 20 questions.
Display the final results in a comparison table to analyze the impact of each prompting strategy.
</p>

In [35]:
def calculate_rouge_scores(predictions, references):

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    all_scores = {
        'rouge1': {'precision': [], 'recall': [], 'fmeasure': []},
        'rouge2': {'precision': [], 'recall': [], 'fmeasure': []},
        'rougeL': {'precision': [], 'recall': [], 'fmeasure': []}
    }
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        
        for metric in ['rouge1', 'rouge2', 'rougeL']:
            all_scores[metric]['precision'].append(scores[metric].precision)
            all_scores[metric]['recall'].append(scores[metric].recall)
            all_scores[metric]['fmeasure'].append(scores[metric].fmeasure)
    
    results = {}
    for metric in ['rouge1', 'rouge2', 'rougeL']:
        results[f'{metric}_precision'] = np.mean(all_scores[metric]['precision'])
        results[f'{metric}_recall'] = np.mean(all_scores[metric]['recall'])
        results[f'{metric}_fmeasure'] = np.mean(all_scores[metric]['fmeasure'])
    
    return results


def evaluate_method(predictions_df, method_name):

    predictions = predictions_df['predicted_answer'].tolist()
    references = predictions_df['gold_answer'].tolist()
    scores = calculate_rouge_scores(predictions, references)
    scores['method'] = method_name
    return scores

no_rag_df = pd.read_csv("no_rag_predictions.csv")
zero_shot_df = pd.read_csv("rag_zero_shot_predictions.csv")
few_shot_df = pd.read_csv("rag_few_shot_predictions.csv")
cot_df = pd.read_csv("rag_cot_predictions.csv")

methods = [
    ("No RAG", no_rag_df),
    ("Zero-shot RAG", zero_shot_df),
    ("Few-shot RAG", few_shot_df),
    ("CoT RAG", cot_df)
]

all_results = []

for method_name, df in methods:
    scores = evaluate_method(df, method_name)
    all_results.append(scores)

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[[
    'method',
    'rouge1_precision', 'rouge1_recall', 'rouge1_fmeasure',
    'rouge2_fmeasure',
    'rougeL_fmeasure'
]]

comparison_df.columns = [
    'Method',
    'R1-P', 'R1-R', 'R1-F1',
    'R2-F1',
    'RL-F1'
]

comparison_df = comparison_df.round(4)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("rouge_comparison_results.csv", index=False)

       Method   R1-P   R1-R  R1-F1  R2-F1  RL-F1
       No RAG 0.0615 0.2475 0.0956 0.0071 0.0918
Zero-shot RAG 0.2072 0.8465 0.3088 0.2050 0.3052
 Few-shot RAG 0.4673 0.8448 0.5096 0.3393 0.5026
      CoT RAG 0.2065 0.8708 0.2972 0.2049 0.2889


---
## 8.1  Analyze ROUGE Scores (2 Points)

> **💬 Question (written answer — ~200 words):**  
> Analyze the ROUGE scores obtained from all four methods (No RAG, Zero-shot, Few-shot, and CoT).  
> - Which method achieved the highest ROUGE scores and why?  
> - How much did RAG improve the results compared to No RAG baseline?  
> - What does the difference between ROUGE-1 and ROUGE-L tell you about the quality of the generated answers?  
> - Based on your results, which prompting strategy would you recommend and why?  

Few-shot RAG got the highest ROUGE score at 0.5096. It uses context and three examples to guide the model, which helps produce more accurate answers.

RAG methods improved results a lot compared to No RAG. Few-shot RAG scored 5.3 times higher. Zero-shot and CoT also went up from 0.0956 to 0.3088 and 0.2972. This shows that context and examples greatly improve answer quality.

ROUGE-1 and ROUGE-L scores are very close across all RAG methods. This means the model keeps good word order and sentence structure while also keeping key content. The answers are both informative and grammatically sound.

Few-shot RAG because it has the highest scores on all measures with ROUGE-1 F1 of 0.5096 and ROUGE-2 F1 of 0.3393. CoT gives reasoning transparency, but Few-shot gives more reliable and accurate answers with better precision and F1 scores.

---
## 9 Evaluation with RoBERTa

<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, evaluate the predicted answers from all four methods (No RAG, Zero-shot, Few-shot, and CoT)
using a <b>RoBERTa-based NLI model</b> loaded from Hugging Face (<b>FacebookAI/roberta-large-mnli</b>) to measure semantic equivalence between the predicted and gold answers.
For each predicted answer, the model classifies the relationship with the gold answer as <b>entailment</b>, <b>neutral</b>, or <b>contradiction</b>.
For each method, compute and display the <b>average percentage</b> of each label across all 20 questions.
Display the final results in a comparison table alongside the ROUGE scores to analyze the impact of each prompting strategy.
</p>

## 9.1 Load RoBERTa NLI (2 Points)

Load the RoBERTa NLI model and tokenizer from Hugging Face (`FacebookAI/roberta-large-mnli`).

In [7]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

nli_model = nli_model.to(device)

print("total params: " ,sum(p.numel() for p in nli_model.parameters()))

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


total params:  355362819


## 9.2  Define NLI Scoring Function (2 Points)

Define a function that takes a list of predicted answers and gold answers, and for each pair classifies the relationship as **entailment**, **neutral**, or **contradiction**.

In [9]:
def classify_nli(predicted, reference, model, tokenizer):
    
    inputs = tokenizer(
        reference,
        predicted,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    probs = softmax(logits.cpu().numpy(), axis=1)[0]
    label_map = {0: "contradiction", 1: "neutral", 2: "entailment"}
    return label_map[np.argmax(probs)], probs


def evaluate_nli(predictions_df, method_name, model, tokenizer):
    labels = []
    for _, row in predictions_df.iterrows():
        label, _ = classify_nli(row['predicted_answer'], row['gold_answer'], model, tokenizer)
        labels.append(label)
    
    counts = {
        'entailment': labels.count('entailment'),
        'neutral': labels.count('neutral'),
        'contradiction': labels.count('contradiction')
    }
    total = len(labels)
    percentages = {k: (v / total) * 100 for k, v in counts.items()}
    
    return {'method': method_name, 'counts': counts, 'percentages': percentages}

## 9.3  Run Evaluation and Display Results (1 Points)

Run the NLI scoring function on all four methods and display the average percentage of each label in a comparison table.
Also merge the NLI results with the ROUGE scores from the previous section into a single final table.

In [10]:
no_rag_df = pd.read_csv("no_rag_predictions.csv")
zero_shot_df = pd.read_csv("rag_zero_shot_predictions.csv")
few_shot_df = pd.read_csv("rag_few_shot_predictions.csv")
cot_df = pd.read_csv("rag_cot_predictions.csv")

methods = [
    ("No RAG", no_rag_df),
    ("Zero-shot RAG", zero_shot_df),
    ("Few-shot RAG", few_shot_df),
    ("CoT RAG", cot_df)
]

nli_results = []
for method_name, df in methods:
    result = evaluate_nli(df, method_name, nli_model, nli_tokenizer)
    nli_results.append(result)

nli_data = []
for result in nli_results:
    nli_data.append({
        'Method': result['method'],
        'Entailment': result['percentages']['entailment'],
        'Neutral': result['percentages']['neutral'],
        'Contradiction': result['percentages']['contradiction']
    })
nli_df = pd.DataFrame(nli_data).round(1)

rouge_df = pd.read_csv("rouge_comparison_results.csv")
final_df = rouge_df.merge(nli_df, on='Method', how='left')
final_df = final_df[['Method', 'R1-F1', 'R2-F1', 'RL-F1', 'Entailment', 'Neutral', 'Contradiction']].round(4)

print(final_df.to_string(index=False))
final_df.to_csv("final_comparison_results.csv", index=False)

       Method  R1-F1  R2-F1  RL-F1  Entailment  Neutral  Contradiction
       No RAG 0.0956 0.0071 0.0918         0.0     70.0           30.0
Zero-shot RAG 0.3088 0.2050 0.3052         0.0     95.0            5.0
 Few-shot RAG 0.5096 0.3393 0.5026        35.0     55.0           10.0
      CoT RAG 0.2972 0.2049 0.2889         5.0     85.0           10.0


---
## 9.4  Analyze RoBERTa NLI Results (4 Points)

> **💬 Question (written answer — ~200 words):**  
> Analyze the RoBERTa NLI results obtained from all four methods (No RAG, Zero-shot, Few-shot, and CoT).  
> - Which method achieved the highest entailment percentage and why?  
> - How does the contradiction rate change between No RAG and RAG-based methods?  
> - Comparing the NLI results with the ROUGE scores, do both metrics agree on which method performs best?  
> - What does a high neutral percentage tell you about the quality of the generated answers?  

a) Few-shot RAG had the highest entailment at 35.0 percent. It uses context and three examples to guide the model. This helps produce answers that match the gold answers in meaning.

b) The contradiction rate dropped from 30.0 percent in No RAG to 5.0 percent in Zero-shot RAG. Few-shot and CoT both had 10.0 percent. This shows context reduces conflicting information. Zero-shot worked best for lowering contradictions.

c) Both metrics agree Few-shot RAG is the best. It has the highest ROUGE scores and entailment at 35.0 percent. CoT scored lower on both. Zero-shot had the lowest contradiction at 5.0 percent. ROUGE and NLI measure different things, with ROUGE looking at word overlap and NLI looking at meaning.

d) A high neutral percentage means answers are not clearly right or wrong. Zero-shot had 95.0 percent neutral, meaning most answers do not match the gold answers. The model gives vague responses that miss specific information. High neutral percentages usually mean poor quality.

---
## 10  Fine-Tuning With LoRA



<p style="text-align: left; padding:30px; background-color:rgb(12, 12, 12); border-radius: 12px; color: white; font-family: monospace;">
In this section, you need to fine-tune the <b>Qwen/Qwen 3.5 2B</b> model using <b>LoRA (Low-Rank Adaptation)</b>.
To avoid <b>Out of Memory</b> errors, please <b>restart your runtime</b> before running this section and reinstall the required libraries from the <b>requirements.txt</b> file.
</p>

## 10.1 Load Fine-Tuning Dataset

Load the fine-tuning datasets from **finetune_train.csv** and **finetune_val.csv** and prepare them for tokenization.

In [11]:
train_df = pd.read_csv(DATA_DIR / "finetune_train.csv")
val_df = pd.read_csv(DATA_DIR / "finetune_val.csv")

## 10.2 Load Model and Tokenizer (2 Points)

Load the **Qwen/Qwen3.5-2B** model and its tokenizer from 🤗 Hugging Face using `AutoModelForCausalLM` and `AutoTokenizer`.

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

if device == "cuda":
    model = model.to(device)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

## 10.3 Tokenize Dataset (4 Points)

Define a tokenization function using the loaded tokenizer and apply it to both the train and validation datasets.

In [13]:
def tokenize_function(examples):
    formatted = []
    for text in examples:
        messages = [
            {"role": "user", "content": text}
        ]
        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True
        )
        formatted.append(formatted_text)
    return tokenizer(formatted, truncation=True, padding=False, max_length=512, return_tensors=None)

train_tokenized = tokenize_function(train_df['text'].tolist())
val_tokenized = tokenize_function(val_df['text'].tolist())

train_dataset = Dataset.from_dict({
    'input_ids': train_tokenized['input_ids'],
    'attention_mask': train_tokenized['attention_mask']
})
val_dataset = Dataset.from_dict({
    'input_ids': val_tokenized['input_ids'],
    'attention_mask': val_tokenized['attention_mask']
})

## 10.4 Fine-Tune with LoRA (8 Points)

Fine-tune the model using **LoRA** with the following configuration:
- `r=8`, `lora_alpha=16`, `lora_dropout=0.05`
- `target_modules`: `q_proj` and `v_proj`
- `num_train_epochs=1`
- `per_device_train_batch_size=2`
- `gradient_accumulation_steps=8`
- `learning_rate=2e-4`
- Enable `fp16` and `gradient_checkpointing` to avoid out of memory errors.

After training, save the fine-tuned model to **./lora_output** or GOOGLE DRIVE for use in the next sections.

In [ ]:
torch.cuda.empty_cache()
gc.collect()

1264

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

model.save_pretrained("./lora_output")
tokenizer.save_pretrained("./lora_output")

/opt/conda/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 417,792 || all params: 1,882,242,880 || trainable%: 0.0222


Step,Training Loss,Validation Loss
50,2.695918,2.488409
100,2.418984,2.394014
150,2.368809,2.384183
200,2.361971,2.382454
250,2.335639,2.381148
286,2.335639,2.381050


('./lora_output/tokenizer_config.json',
 './lora_output/chat_template.jinja',
 './lora_output/tokenizer.json')

In [31]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32,expandable_segments:True"

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

## 10.5 Generate Answers with Fine-Tuned Model (2 Points)

Using the fine-tuned model, generate answers for the 20 questions in **squad_gold_20.csv** without providing any context (No RAG).
Store the predicted answers alongside the gold answers in a CSV file.

In [33]:

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model, "./lora_output")
model = model.merge_and_unload()
model.eval()


ft_results = []

for idx, row in gold_df.iterrows():
    question = row["question"]
    gold_answer = row["answer"]
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the following question accurately and concisely. IMPORTANT: Provide only a SHORT, CONCISE answer with entire response as MAXIMUM 20 WORDS. Do not add explanations or context."},
        {"role": "user", "content": question}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
            top_p=0.8,
            top_k=20,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_answer = full_response.replace(prompt, "").strip()
    predicted_answer = predicted_answer.split("</think>")[-1].strip()
    
        
    print(f"[{idx+1}/{len(gold_df)}] Q: {question}")
    print(f"Gold: {gold_answer}")
    print(f"Pred: {predicted_answer}\n")
    print("------------\n")
    
    ft_results.append({
        "id": row["id"],
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": predicted_answer
    })

ft_df = pd.DataFrame(ft_results)
ft_df.to_csv("finetuned_predictions.csv", index=False)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[1/20] Q: When is the last time a fumble return touchdown happened in a Super Bowl?
Gold: Super Bowl XXVIII
Pred: The last fumble-return TD was in Super Bowl XXXVIII (1984), where it occurred during the game against the New England Patriots on

------------

[2/20] Q: According to the AAA, what is the percentage of the gas stations that ran out of gasoline?
Gold: last week of February 1974,
Pred: AAA reports no nationwide data on this specific statistic; however, some states report significant shortages due to supply chain disruptions.

------------

[3/20] Q: What choice did French have for surrendering land?
Gold: continental North American possessions east of the Mississippi or the Caribbean islands of Guadeloupe and Martinique
Pred: France chose to negotiate peace terms rather than immediately surrender all territory after World War I's defeat in France.

------------

[4/20] Q: Which articles state that the member states' rights to deliver public services may not be obstructed?
Go

## 10.6 Evaluate with ROUGE (2 Points)

Compute **ROUGE-1**, **ROUGE-2**, and **ROUGE-L** scores for the fine-tuned model's predictions using the **rouge-score** library.
Display the average scores across all 20 questions.

In [34]:
ft_df = pd.read_csv("finetuned_predictions.csv")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

ft_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for _, row in ft_df.iterrows():
    pred = row['predicted_answer'].strip() or "no answer"
    scores = scorer.score(row['gold_answer'], pred)
    for metric in ['rouge1', 'rouge2', 'rougeL']:
        ft_scores[metric].append(scores[metric].fmeasure)

ft_rouge = {
    'R1-F1': np.mean(ft_scores['rouge1']),
    'R2-F1': np.mean(ft_scores['rouge2']),
    'RL-F1': np.mean(ft_scores['rougeL'])
}

print(f"Fine-tuned ROUGE-1: {ft_rouge['R1-F1']:.4f}")
print(f"Fine-tuned ROUGE-2: {ft_rouge['R2-F1']:.4f}")
print(f"Fine-tuned ROUGE-L: {ft_rouge['RL-F1']:.4f}")

Fine-tuned ROUGE-1: 0.0901
Fine-tuned ROUGE-2: 0.0129
Fine-tuned ROUGE-L: 0.0901


## 10.7 Load RoBERTa NLI Model (1 Points)

Load the **FacebookAI/roberta-large-mnli** model and its tokenizer from 🤗 Hugging Face.

In [35]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

nli_model = nli_model.to(device)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 10.8 Define RoBERTa NLI Scoring Function (1 Points)

Define a function that takes a list of predicted answers and gold answers, and for each pair classifies the relationship as **entailment**, **neutral**, or **contradiction**.

In [37]:
def classify_nli(predicted, reference, model, tokenizer):
    
    inputs = tokenizer(reference, predicted, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = softmax(outputs.logits.cpu().numpy(), axis=1)[0]
    
    label_map = {0: "contradiction", 1: "neutral", 2: "entailment"}
    return label_map[np.argmax(probs)], probs


def evaluate_nli(predictions_df, method_name, model, tokenizer):
    labels = []
    for _, row in predictions_df.iterrows():
        label, _ = classify_nli(row['predicted_answer'], row['gold_answer'], model, tokenizer)
        labels.append(label)
    
    counts = {
        'entailment': labels.count('entailment'),
        'neutral': labels.count('neutral'),
        'contradiction': labels.count('contradiction')
    }
    total = len(labels)
    percentages = {k: (v / total) * 100 for k, v in counts.items()}
    return {'method': method_name, 'counts': counts, 'percentages': percentages}

## 10.9 Evaluate Fine-Tuned Model with RoBERTa NLI (1 Points)

Run the NLI scoring function on the fine-tuned model's predictions and display the average percentage of each label across all 20 questions.

In [ ]:
ft_df = pd.read_csv("finetuned_predictions.csv")
ft_nli = evaluate_nli(ft_df, "Fine-tuned", nli_model, nli_tokenizer)
final_df = pd.read_csv("final_comparison_results.csv")

ft_row = {
    'Method': 'Fine-tuned',
    'R1-F1': ft_rouge['R1-F1'],
    'R2-F1': ft_rouge['R2-F1'],
    'RL-F1': ft_rouge['RL-F1'],
    'Entailment': ft_nli['percentages']['entailment'],
    'Neutral': ft_nli['percentages']['neutral'],
    'Contradiction': ft_nli['percentages']['contradiction']
}

final_with_ft = pd.concat([final_df, pd.DataFrame([ft_row])], ignore_index=True).round(4)

print(final_with_ft.to_string(index=False))
final_with_ft.to_csv("final_comparison_results.csv", index=False)

       Method  R1-F1  R2-F1  RL-F1  Entailment  Neutral  Contradiction
       No RAG 0.0956 0.0071 0.0918         0.0     70.0           30.0
Zero-shot RAG 0.3088 0.2050 0.3052         0.0     95.0            5.0
 Few-shot RAG 0.5096 0.3393 0.5026        35.0     55.0           10.0
      CoT RAG 0.2972 0.2049 0.2889         5.0     85.0           10.0
   Fine-tuned 0.0901 0.0129 0.0901         5.0     80.0           15.0


---
## 10.10  RAG vs Fine-Tuned Model Analysis (8 Points)

> **💬 Question (written answer — ~200 words):**  
> Compare the results of the RAG-based methods (Zero-shot, Few-shot, and CoT) with the fine-tuned model (No RAG) using both ROUGE and RoBERTa NLI scores.  
> - Which approach achieved better results overall and why?   
> - What are the limitations of fine-tuning with LoRA on a small dataset compared to RAG?  
> - In what scenarios would you prefer fine-tuning over RAG?  

a) RAG methods beat the fine tuned model on all metrics. few shot RAG was best with ROUGE-1 F1 of 0.51 and 35% entailment. The fine-tuned model scored only 0.09 and 5 pervent. All RAG methods outperformed it clearly. This shows retrieval works better than fine-tuning for QA.

b) in fine tune model dataset had only 4500 samples, which is too small. The data lacked proper QA pairs, so it was more like language modeling. RAG uses context at inference time and does not need to memorize facts, so it works well with small data.

c) Fine-tuning is better when retrieval is not feasible, or when fast inference is needed. But for general QA with large knowledge, RAG is better.